# Stage 2 bus structural factorial (`gs_s2`)

This notebook analyses `configs/gnn_graph_screening/stage2_structure`: a
complete `2 x 2 x 2` structural factorial on the bus graph with three seeds
each (24 runs).

The three screened factors are:

- **`e`** -- `gnn_add_substation_edges`: bidirectional edges between busbars of
  the same substation;
- **`n`** -- `gnn_add_substation_nodes`: explicit substation summary nodes;
- **`v`** -- `gnn_readout_aggr == "virtual_node"`: virtual-node readout instead
  of mean pooling.

Everything else is held constant (bus graph, no preprocessing, GINE actor, MLP
critic, 15M steps). The baseline is the unaugmented **`e0n0v0`**.

Because this is a `2^3` design rather than a 2-D grid, the heatmaps use
`(e, n)` rows against `v` columns -- which shows all eight cells with no
marginalising -- plus the three pairwise marginals and an explicit main-effect
table.

In [48]:
from pathlib import Path
import importlib
import sys
import tomllib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError(
        "Could not locate Topology_Task/analysis/metrics/helpers"
    )

import wandb_metrics as wm
wm = importlib.reload(wm)
import survival_comparison as sc
sc = importlib.reload(sc)
print("wandb_metrics:", wm.__file__)
print("survival_comparison:", sc.__file__)
print("task directory:", wm.TASK_DIR)
print("done")

wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py
survival_comparison: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/survival_comparison.py
task directory: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
done


## Analysis controls

`COMPARISON_BUDGET_STEPS = None` compares every run at the largest step all
selected runs reached.

> **Walltime warning.** These configs set `time_limit = 5760` minutes (96 h),
> but the SLURM allocations are shorter: `job_jed.sh` requests 5040 min (84 h)
> and `job_izar.sh` requests 4332 min (72.2 h). The in-loop guard in
> `alg/mappo/core.py` therefore never fires -- SLURM ends the job first -- and
> the two clusters get materially different effective walltimes. Expect
> truncated, backend-correlated endpoints, and check the coverage section
> before reading anything into the endpoint numbers.

In [49]:
USE_LOCAL_CACHE_ONLY = True
SMOOTH_WINDOW = 5
HEATMAP_LAST_N_TEST_EVALS = 7
TARGET_BUDGET_STEPS = 15_000_000
COMPARISON_BUDGET_STEPS = None

S2_DIR = (
    wm.TASK_DIR / "configs" / "gnn_graph_screening" / "stage2_structure"
)
S2_MANIFEST = S2_DIR / "manifest.csv"
S2_RUN_PREFIX = "gs_s2_bus_n0_none"

BASELINE_STRUCTURE = "e0n0v0"

# Factor -> (catalog column, off label, on label)
STRUCTURE_FACTORS = {
    "e": ("substation_edges", "e0 (no substation edges)", "e1 (substation edges)"),
    "n": ("substation_nodes", "n0 (no substation nodes)", "n1 (substation nodes)"),
    "v": ("virtual_node", "v0 (mean readout)", "v1 (virtual-node readout)"),
}
FACTOR_COLUMNS = ["substation_edges", "substation_nodes", "virtual_node"]

EN_ORDER = ["e0n0", "e0n1", "e1n0", "e1n1"]
V_ORDER = ["v0", "v1"]
STRUCTURE_ORDER = [f"{en}{v}" for en in EN_ORDER for v in V_ORDER]

print("config folder:", S2_DIR)
print("baseline structure:", BASELINE_STRUCTURE)
print("structure order:", STRUCTURE_ORDER)
print("done")

config folder: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/configs/gnn_graph_screening/stage2_structure
baseline structure: e0n0v0
structure order: ['e0n0v0', 'e0n0v1', 'e0n1v0', 'e0n1v1', 'e1n0v0', 'e1n0v1', 'e1n1v0', 'e1n1v1']
done


## Build the run catalog from TOML

Factor values are read from the config files, not parsed out of the file names,
so the `e`/`n`/`v` codes are derived from what the runs actually did. The
cluster assignment is taken from the folder's `manifest.csv`, which records the
designed JED/Izar split.

In [50]:
def structure_record(path):
    with path.open("rb") as file:
        config = tomllib.load(file)
    args = config["args"]
    run = config.get("run", {})
    readout = str(args.get("gnn_readout_aggr", "mean"))
    record = {
        "config_path": str(path.relative_to(wm.TASK_DIR)),
        "config": path.name,
        "run_name": str(run.get("name", path.stem)),
        "seed": int(args.get("seed", 0)),
        "declared_cuda": bool(args.get("cuda", False)),
        "graph_type": str(args.get("gnn_graph_type", "bus")),
        "encoder": str(args.get("gnn_type", "gine")),
        "readout": readout,
        "substation_edges": bool(args.get("gnn_add_substation_edges", False)),
        "substation_nodes": bool(args.get("gnn_add_substation_nodes", False)),
        "virtual_node": readout == "virtual_node",
        "summary_direction": str(
            args.get("gnn_summary_edge_direction", "bidirectional")
        ),
        "configured_steps": int(args.get("total_timesteps", TARGET_BUDGET_STEPS)),
        "time_limit_minutes": float(args.get("time_limit", np.nan)),
    }
    record["structure"] = "e{}n{}v{}".format(
        int(record["substation_edges"]),
        int(record["substation_nodes"]),
        int(record["virtual_node"]),
    )
    record["en_label"] = record["structure"][:4]
    record["v_label"] = record["structure"][4:]
    record["is_baseline"] = record["structure"] == BASELINE_STRUCTURE
    for code_key, (column, off_label, on_label) in STRUCTURE_FACTORS.items():
        record[f"{code_key}_facet"] = (
            on_label if record[column] else off_label
        )
    return record


if not S2_DIR.exists():
    raise FileNotFoundError(f"Missing config folder: {S2_DIR}")

config_catalog = pd.DataFrame(
    [structure_record(path) for path in sorted(S2_DIR.glob("*.toml"))]
)
if config_catalog.empty:
    raise RuntimeError(f"No TOML files were found in {S2_DIR}.")

# Designed cluster assignment, when the manifest is available.
if S2_MANIFEST.exists():
    manifest = pd.read_csv(S2_MANIFEST)
    if "cluster" in manifest and "config" in manifest:
        config_catalog = config_catalog.merge(
            manifest[["config", "cluster"]], on="config", how="left"
        )
        print("cluster assignment from manifest.csv:")
        print(config_catalog["cluster"].value_counts().to_string())
    else:
        config_catalog["cluster"] = np.nan
else:
    print(f"No manifest at {S2_MANIFEST}; cluster column left empty.")
    config_catalog["cluster"] = np.nan

print(f"\nCataloged {len(config_catalog)} declared runs.")

held_constant = [
    "graph_type",
    "encoder",
    "summary_direction",
    "configured_steps",
    "time_limit_minutes",
]
for column in held_constant:
    values = sorted(config_catalog[column].astype(str).unique())
    flag = "" if len(values) == 1 else "  <-- NOT CONSTANT"
    print(f"  {column}: {values}{flag}")

missing_structures = sorted(set(STRUCTURE_ORDER) - set(config_catalog["structure"]))
if missing_structures:
    print("Missing structures:", missing_structures)

display(
    config_catalog.pivot_table(
        index="en_label",
        columns="v_label",
        values="seed",
        aggfunc="count",
        fill_value=0,
    ).reindex(index=EN_ORDER, columns=V_ORDER)
)
print("done")

cluster assignment from manifest.csv:
cluster
jed     12
izar    12

Cataloged 24 declared runs.
  graph_type: ['bus']
  encoder: ['gine']
  summary_direction: ['bidirectional']
  configured_steps: ['15000000']
  time_limit_minutes: ['5760.0']


v_label,v0,v1
en_label,,
e0n0,3,3
e0n1,3,3
e1n0,3,3
e1n1,3,3


done


## Load the W&B histories

`compute_backend` comes from the effective downloaded run configuration
(`cuda=True` implies the Izar GPU cluster), which is then cross-checked against
the designed assignment in `manifest.csv`.

In [51]:
requested_run_names = config_catalog["run_name"].drop_duplicates().tolist()
requested_run_name_set = set(requested_run_names)
wm.configure_run_filter_from_names(requested_run_names)
data = wm.load_wandb_data(use_local_cache_only=USE_LOCAL_CACHE_ONLY)

runs_df = data.runs_df.copy()
history_df = data.history_df.copy()
if not history_df.empty:
    history_df = history_df[
        history_df["run_name"].astype(str).isin(requested_run_name_set)
    ].copy()
if "name" in runs_df:
    runs_df = runs_df[
        runs_df["name"].astype(str).isin(requested_run_name_set)
    ].copy()


def parse_optional_bool(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {"true", "1", "yes", "y"}:
            return True
        if normalized in {"false", "0", "no", "n"}:
            return False
    return bool(value)


if "cuda" in runs_df and "name" in runs_df:
    runtime_backend = runs_df[["name", "cuda"]].copy()
    runtime_backend["runtime_cuda"] = runtime_backend["cuda"].map(
        parse_optional_bool
    )
    runtime_backend = (
        runtime_backend.dropna(subset=["runtime_cuda"])
        .drop_duplicates("name", keep="last")
        .rename(columns={"name": "run_name"})[["run_name", "runtime_cuda"]]
    )
    config_catalog = config_catalog.merge(
        runtime_backend, on="run_name", how="left"
    )
else:
    config_catalog["runtime_cuda"] = np.nan
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].where(
    config_catalog["runtime_cuda"].notna(), config_catalog["declared_cuda"]
)
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].astype(bool)
config_catalog["compute_backend"] = np.where(
    config_catalog["runtime_cuda"], "IZAR (GPU)", "JED (CPU)"
)

# The manifest says where each run was meant to go; cuda says where it ran.
if config_catalog["cluster"].notna().any():
    expected = np.where(
        config_catalog["cluster"].astype(str).str.lower().eq("izar"),
        "IZAR (GPU)",
        "JED (CPU)",
    )
    mismatch = config_catalog[config_catalog["compute_backend"] != expected]
    if len(mismatch):
        print(
            f"{len(mismatch)} run(s) ran on a different cluster than the "
            "manifest assigned:"
        )
        display(
            mismatch[["run_name", "cluster", "compute_backend"]]
        )
    else:
        print("Backend matches the manifest assignment for every run.")

found_run_names = set(history_df.get("run_name", pd.Series(dtype=str)))
print(
    f"Loaded histories for {len(found_run_names)} / "
    f"{len(requested_run_names)} declared runs."
)
missing_run_names = sorted(requested_run_name_set - found_run_names)
if missing_run_names:
    print("Missing histories:")
    for name in missing_run_names:
        print("  ", name)
print("done")

Explicit run-name filter: 24 candidates
Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 24 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    24
History artifact setup: local_only=True, runs_df=24
[ 1/24] loading artifact cache: gs_s2_bus_n0_none_e0n0v0_s0
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s2/runs/gs_s2_bus_n0_none_e0n0v0_s0__MAPPO_bus14_T_0_0__I__1785074341_38855/history.parquet in 0.1s
[ 2/24] loading artifact cache: gs_s2_bus_n0_none_e0n0v0_s1
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s2/runs/gs_s2_bus

/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py:626: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(pieces, ignore_index=True, sort=False)


    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s2/runs/gs_s2_bus_n0_none_e0n0v1_s1__MAPPO_bus14_T_1_0__I__1785060732_6190/history.parquet in 0.1s
[ 6/24] loading artifact cache: gs_s2_bus_n0_none_e0n0v1_s2
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s2/runs/gs_s2_bus_n0_none_e0n0v1_s2__MAPPO_bus14_T_2_0__I__1785082405_13123/history.parquet in 0.0s
[ 7/24] loading artifact cache: gs_s2_bus_n0_none_e0n1v0_s0
    loaded 359 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s2/runs/gs_s2_bus_n0_none_e0n1v0_s0__MAPPO_bus14_T_0_0__I__1785060732_48864/history.parquet in 0.0s
[ 8/24] loading artifact cache: gs_s2_bus_n0_none_e0n1v0_s1
    loaded 361 rows, 113 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s2/runs/gs_s2_bus_n0_none_e0n1v0_s1__MAPPO_bus14_T_1_

## Coverage

Read this before the endpoint sections. Given the walltime mismatch noted
above, a `completion_pct` well below 100 is expected, and a systematic gap
between JED and Izar means the backend is confounded with training length.

In [52]:
if history_df.empty:
    raise RuntimeError(
        "No history was loaded. Download the gs_s2 runs first, or set "
        "USE_LOCAL_CACHE_ONLY = False."
    )

observed_progress = (
    history_df.groupby("run_name", as_index=False)["step"]
    .max()
    .rename(columns={"step": "observed_steps"})
)
coverage = config_catalog.merge(observed_progress, on="run_name", how="left")
coverage["observed_steps_m"] = coverage["observed_steps"] / 1_000_000
coverage["completion_pct"] = (
    100 * coverage["observed_steps"] / coverage["configured_steps"]
)
coverage["has_history"] = coverage["observed_steps"].notna()

analysis_catalog = coverage[coverage["has_history"]].copy()

with pd.option_context("display.max_colwidth", None):
    display(
        coverage.sort_values(["structure", "seed"])[
            [
                "structure",
                "seed",
                "run_name",
                "cluster",
                "compute_backend",
                "observed_steps_m",
                "completion_pct",
                "has_history",
            ]
        ].round(2)
    )

print("Observed-step spread by backend:")
display(
    analysis_catalog.groupby("compute_backend", as_index=False)
    .agg(
        runs=("run_name", "nunique"),
        min_steps_m=("observed_steps_m", "min"),
        median_steps_m=("observed_steps_m", "median"),
        max_steps_m=("observed_steps_m", "max"),
    )
    .round(2)
)

progress_plot = px.bar(
    coverage.sort_values("observed_steps_m"),
    x="observed_steps_m",
    y="run_name",
    color="compute_backend",
    orientation="h",
    hover_data=["structure", "seed", "cluster"],
    title="gs_s2 run coverage",
    labels={
        "observed_steps_m": "Observed environment steps (millions)",
        "run_name": "Run",
    },
    height=max(600, 22 * len(coverage)),
)
progress_plot.add_vline(
    x=TARGET_BUDGET_STEPS / 1_000_000,
    line_dash="dash",
    annotation_text="15M target",
)
progress_plot.show()
print("done")

,structure,seed,run_name,cluster,compute_backend,observed_steps_m,completion_pct,has_history
0,e0n0v0,0,gs_s2_bus_n0_none_e0n0v0_s0,jed,JED (CPU),14.97,99.81,True
1,e0n0v0,1,gs_s2_bus_n0_none_e0n0v0_s1,izar,IZAR (GPU),14.97,99.81,True
2,e0n0v0,2,gs_s2_bus_n0_none_e0n0v0_s2,izar,IZAR (GPU),14.97,99.81,True
3,e0n0v1,0,gs_s2_bus_n0_none_e0n0v1_s0,jed,JED (CPU),14.97,99.81,True
4,e0n0v1,1,gs_s2_bus_n0_none_e0n0v1_s1,izar,IZAR (GPU),14.97,99.81,True
5,e0n0v1,2,gs_s2_bus_n0_none_e0n0v1_s2,jed,JED (CPU),14.97,99.81,True
6,e0n1v0,0,gs_s2_bus_n0_none_e0n1v0_s0,izar,IZAR (GPU),14.89,99.26,True
7,e0n1v0,1,gs_s2_bus_n0_none_e0n1v0_s1,jed,JED (CPU),14.97,99.81,True
8,e0n1v0,2,gs_s2_bus_n0_none_e0n1v0_s2,jed,JED (CPU),14.97,99.81,True
9,e0n1v1,0,gs_s2_bus_n0_none_e0n1v1_s0,izar,IZAR (GPU),14.97,99.81,True


Observed-step spread by backend:


,compute_backend,runs,min_steps_m,median_steps_m,max_steps_m
0,IZAR (GPU),12,14.72,14.97,14.97
1,JED (CPU),12,14.97,14.97,14.97


done


## Extract test episodic-survival curves

In [53]:
survival_long = sc.extract_survival_curves(
    history_df,
    catalog=analysis_catalog,
    split="test",
    smooth=1,
)
if survival_long.empty:
    raise RuntimeError("No test episodic-survival history was found.")

print("metrics used:", sorted(survival_long["metric"].unique()))
print(f"{survival_long['run_name'].nunique()} runs with survival curves")

max_survival_steps = survival_long.groupby("run_name")["step"].max()
automatic_common_budget = int(max_survival_steps.min())
comparison_budget = int(
    COMPARISON_BUDGET_STEPS
    if COMPARISON_BUDGET_STEPS is not None
    else automatic_common_budget
)
print(
    "Comparison budget:",
    f"{comparison_budget / 1_000_000:.3f}M steps",
    "(automatic common budget)"
    if COMPARISON_BUDGET_STEPS is None
    else "(user selected)",
)
if COMPARISON_BUDGET_STEPS is None:
    print(f"  set by the shortest run: {max_survival_steps.idxmin()}")
print("done")

metrics used: ['test/charts/episodic_survival']
24 runs with survival curves
Comparison budget: 14.681M steps (automatic common budget)
  set by the shortest run: gs_s2_bus_n0_none_e1n1v0_s2
done


## Per-run endpoint statistics

In [54]:
def endpoint_at_budget(frame, budget):
    if frame["step"].max() < budget:
        return np.nan
    eligible = frame[frame["step"] <= budget]
    if eligible.empty:
        return np.nan
    smoothed = (
        eligible["raw_survival_pct"]
        .rolling(SMOOTH_WINDOW, min_periods=1)
        .mean()
    )
    return float(smoothed.iloc[-1])


endpoint_rows = []
for run_name, frame in survival_long.groupby("run_name", sort=False):
    frame = frame.sort_values("step")
    tail = frame.tail(HEATMAP_LAST_N_TEST_EVALS)
    endpoint_rows.append(
        {
            "run_name": run_name,
            "last_eval_step": frame["step"].max(),
            "n_test_evals": len(frame),
            "last_n_test_evals": len(tail),
            "last_n_test_survival_pct": float(tail["raw_survival_pct"].mean()),
            "comparison_survival_pct": endpoint_at_budget(
                frame, comparison_budget
            ),
            "target_survival_pct": endpoint_at_budget(
                frame, TARGET_BUDGET_STEPS
            ),
        }
    )

endpoint_df = analysis_catalog.merge(
    pd.DataFrame(endpoint_rows), on="run_name", how="left"
)
endpoint_df["last_eval_step_m"] = endpoint_df["last_eval_step"] / 1_000_000

with pd.option_context("display.max_colwidth", None):
    display(
        endpoint_df.sort_values(
            "last_n_test_survival_pct", ascending=False, na_position="last"
        )[
            [
                "structure",
                "seed",
                "compute_backend",
                "last_eval_step_m",
                "last_n_test_evals",
                "last_n_test_survival_pct",
                "comparison_survival_pct",
                "config_path",
            ]
        ].round(2)
    )
print("done")

,structure,seed,compute_backend,last_eval_step_m,last_n_test_evals,last_n_test_survival_pct,comparison_survival_pct,config_path
13,e1n0v0,1,IZAR (GPU),14.93,7,98.59,97.97,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e1n0v0_s1.toml
12,e1n0v0,0,JED (CPU),14.93,7,98.55,98.70,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e1n0v0_s0.toml
1,e0n0v0,1,IZAR (GPU),14.93,7,97.73,98.70,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v0_s1.toml
15,e1n0v1,0,JED (CPU),14.93,7,97.67,96.24,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e1n0v1_s0.toml
5,e0n0v1,2,JED (CPU),14.93,7,96.90,96.36,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v1_s2.toml
3,e0n0v1,0,JED (CPU),14.93,7,96.53,96.10,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v1_s0.toml
0,e0n0v0,0,JED (CPU),14.93,7,95.82,94.76,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v0_s0.toml
8,e0n1v0,2,JED (CPU),14.93,7,95.40,99.27,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n1v0_s2.toml
7,e0n1v0,1,JED (CPU),14.93,7,93.74,90.81,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n1v0_s1.toml
6,e0n1v0,0,IZAR (GPU),14.85,7,93.65,93.82,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n1v0_s0.toml


done


## Aggregated survival curves

One curve per structure, aggregated over its three seeds. Thin lines are the
individual seeds and the band is one standard deviation. The `e0n0v0` baseline
is drawn in grey.

In [55]:
STRUCTURE_COLORS = {
    "e0n0v0": "#6b7280",
    "e0n0v1": "#1f77b4",
    "e0n1v0": "#2ca02c",
    "e0n1v1": "#17becf",
    "e1n0v0": "#d62728",
    "e1n0v1": "#ff7f0e",
    "e1n1v0": "#9467bd",
    "e1n1v1": "#8c564b",
}

absolute_figure = sc.plot_survival_comparison(
    survival_long,
    group_by="structure",
    label_by="structure",
    smooth=SMOOTH_WINDOW,
    uncertainty="std",
    min_members=1,
    show_members=True,
    colors=STRUCTURE_COLORS,
    highlight=[BASELINE_STRUCTURE],
    budget_step=comparison_budget,
    title="gs_s2: test episodic survival by structural configuration",
    width=1450,
    height=700,
)
absolute_figure.show()
print("done")

done


### Small multiples, one factor at a time

Each figure fixes one factor across its panels, so the remaining structures act
as the curves inside each panel. If the same ordering holds in both panels of a
figure, that factor is behaving like an independent main effect.

In [56]:
for code_key, (column, off_label, on_label) in STRUCTURE_FACTORS.items():
    facet_column = f"{code_key}_facet"
    figure = sc.plot_survival_comparison(
        survival_long,
        group_by="structure",
        label_by="structure",
        facet_by=facet_column,
        facet_order=[off_label, on_label],
        smooth=SMOOTH_WINDOW,
        uncertainty="std",
        min_members=1,
        colors=STRUCTURE_COLORS,
        budget_step=comparison_budget,
        title=f"gs_s2: structures split by factor {code_key} ({column})",
        width=1450,
        height=520,
        ncols=2,
    )
    figure.show()
print("done")

done


### One panel per structure, each against the baseline

The editable `wm.plot_run_mean_groups` layout: one subplot per non-baseline
structure, each against `e0n0v0`. Thin lines are seeds, the thick line is the
seed mean, and the band is one standard deviation.

`S2_SUBPLOTS` is a plain dict, so panels can be removed, reordered, or added by
hand. Each entry accepts the usual selectors -- `prefix`, `name`, `runs`,
`contains`, `regex`, `run_dir` -- plus `label`, `color`, and `width`.

In [58]:
def s2_run_prefix(structure):
    """Prefix shared by the three seeds of one structure."""
    return f"{S2_RUN_PREFIX}_{structure}_s"


S2_BASELINE_SPEC = {
    "prefix": s2_run_prefix(BASELINE_STRUCTURE),
    "label": BASELINE_STRUCTURE,
    "color": "#6b7280",
    "width": 4,
}

# Built from STRUCTURE_ORDER so a renamed or reseeded config cannot silently
# drop a panel. Edit the dict afterwards to drop or reorder panels.
S2_SUBPLOTS = {}
for structure in STRUCTURE_ORDER:
    if structure == BASELINE_STRUCTURE:
        continue
    S2_SUBPLOTS[f"{BASELINE_STRUCTURE}  vs  {structure}"] = [
        S2_BASELINE_SPEC,
        {
            "prefix": s2_run_prefix(structure),
            "label": structure,
            "color": STRUCTURE_COLORS.get(structure, "#d62728"),
        },
    ]

print(f"{len(S2_SUBPLOTS)} panels requested")

S2_MEAN_GROUPS = {
    title: wm.resolve_named_plot_specs(run_specs, history=history_df)
    for title, run_specs in S2_SUBPLOTS.items()
}
# A panel with fewer than two resolved specs would show the baseline alone,
# which reads as a real comparison but is not one.
dropped = [title for title, specs in S2_MEAN_GROUPS.items() if len(specs) < 2]
S2_MEAN_GROUPS = {
    title: specs for title, specs in S2_MEAN_GROUPS.items() if len(specs) >= 2
}
if dropped:
    print(f"Dropped {len(dropped)} panel(s) with missing runs:")
    for title in dropped:
        print("  ", title)
print(f"{len(S2_MEAN_GROUPS)} panels plotted")

s2_structure_fig = wm.plot_run_mean_groups(
    S2_MEAN_GROUPS,
    split="test",
    smooth=SMOOTH_WINDOW,
    title="gs_s2: each structural configuration against the e0n0v0 baseline",
    title_font_size=26,
    subplot_title_font_size=20,
    ncols=4,
    subplot_height=470,
    width=2400,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name="gs_s2_structure_vs_baseline_subplots",
    history=history_df,
    horizontal_spacing=0.045,
    vertical_spacing=0.09,
    margin={"l": 50, "r": 20, "t": 90, "b": 45},
)
# Explicit show(): the cell ends with print("done"), so a bare expression here
# would not be auto-displayed.
s2_structure_fig.show()
print("done")

7 panels requested
7 panels plotted
Plot source folders used to build curves: 1 folder(s), 24 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gs_s2 (24 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/gs_s2_structure_vs_baseline_subplots.html


done


## Structure heatmaps

Cell statistic: for each run, the mean of its final
`HEATMAP_LAST_N_TEST_EVALS` **unsmoothed** test evaluations; runs sharing a
cell are then averaged so every run has equal weight.

The first heatmap is the complete factorial -- `(e, n)` rows against `v`
columns, eight cells, nothing marginalised -- followed by its
baseline-relative version, then coverage/spread diagnostics.

In [59]:
HEATMAP_VALUE_COLUMN = "last_n_test_survival_pct"
heatmap_endpoint_df = (
    endpoint_df[
        endpoint_df["last_n_test_evals"] >= HEATMAP_LAST_N_TEST_EVALS
    ]
    .dropna(subset=[HEATMAP_VALUE_COLUMN])
    .copy()
)
excluded = len(endpoint_df) - len(heatmap_endpoint_df)
if excluded:
    print(
        f"Excluded {excluded} run(s) with fewer than "
        f"{HEATMAP_LAST_N_TEST_EVALS} test evaluations."
    )


def structure_table(frame, value_column, aggfunc="mean"):
    table = frame.pivot_table(
        index="en_label",
        columns="v_label",
        values=value_column,
        aggfunc=aggfunc,
    )
    return table.reindex(index=EN_ORDER, columns=V_ORDER).rename_axis(
        index="substation edges / nodes", columns="readout"
    )


def show_heatmap(
    table,
    title,
    color_label,
    x_label,
    y_label,
    colorscale="Viridis",
    zmin=None,
    zmax=None,
    zmid=None,
    text_format=".1f",
    source_df=None,
):
    if table.empty or table.notna().sum().sum() == 0:
        print(f"No data available for: {title}")
        return None
    print(title)
    display(table.round(2))
    kwargs = dict(
        text_auto=text_format,
        aspect="auto",
        color_continuous_scale=colorscale,
        labels={"x": x_label, "y": y_label, "color": color_label},
        title=title,
    )
    if zmin is not None:
        kwargs["zmin"] = zmin
    if zmax is not None:
        kwargs["zmax"] = zmax
    if zmid is not None:
        kwargs["color_continuous_midpoint"] = zmid
    figure = px.imshow(table, **kwargs)
    figure.show()
    if source_df is not None:
        detail = (
            source_df[
                [
                    "structure",
                    "seed",
                    "compute_backend",
                    "run_name",
                    "config_path",
                    HEATMAP_VALUE_COLUMN,
                ]
            ]
            .sort_values(["structure", "seed"])
            .reset_index(drop=True)
        )
        print(f"Configs used for: {title}")
        with pd.option_context("display.max_colwidth", None):
            display(detail.round(2))
    return figure


absolute_table = structure_table(heatmap_endpoint_df, HEATMAP_VALUE_COLUMN)
show_heatmap(
    absolute_table,
    "Full structural factorial — mean final-"
    f"{HEATMAP_LAST_N_TEST_EVALS} test survival",
    "Mean survival (%)",
    "Readout (v)",
    "Substation edges / nodes (e, n)",
    colorscale="Viridis",
    zmin=0,
    zmax=100,
    source_df=heatmap_endpoint_df,
)
print("done")

Full structural factorial — mean final-7 test survival


readout,v0,v1
substation edges / nodes,,
e0n0,94.26,94.56
e0n1,94.26,65.20
e1n0,96.61,89.68
e1n1,85.02,66.53


Configs used for: Full structural factorial — mean final-7 test survival


,structure,seed,compute_backend,run_name,config_path,last_n_test_survival_pct
0,e0n0v0,0,JED (CPU),gs_s2_bus_n0_none_e0n0v0_s0,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v0_s0.toml,95.82
1,e0n0v0,1,IZAR (GPU),gs_s2_bus_n0_none_e0n0v0_s1,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v0_s1.toml,97.73
2,e0n0v0,2,IZAR (GPU),gs_s2_bus_n0_none_e0n0v0_s2,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v0_s2.toml,89.24
3,e0n0v1,0,JED (CPU),gs_s2_bus_n0_none_e0n0v1_s0,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v1_s0.toml,96.53
4,e0n0v1,1,IZAR (GPU),gs_s2_bus_n0_none_e0n0v1_s1,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v1_s1.toml,90.24
5,e0n0v1,2,JED (CPU),gs_s2_bus_n0_none_e0n0v1_s2,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n0v1_s2.toml,96.90
6,e0n1v0,0,IZAR (GPU),gs_s2_bus_n0_none_e0n1v0_s0,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n1v0_s0.toml,93.65
7,e0n1v0,1,JED (CPU),gs_s2_bus_n0_none_e0n1v0_s1,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n1v0_s1.toml,93.74
8,e0n1v0,2,JED (CPU),gs_s2_bus_n0_none_e0n1v0_s2,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n1v0_s2.toml,95.40
9,e0n1v1,0,IZAR (GPU),gs_s2_bus_n0_none_e0n1v1_s0,configs/gnn_graph_screening/stage2_structure/gs_s2_bus_n0_none_e0n1v1_s0.toml,65.41


done


In [60]:
baseline_en = BASELINE_STRUCTURE[:4]
baseline_v = BASELINE_STRUCTURE[4:]
baseline_value = absolute_table.loc[baseline_en, baseline_v]
if pd.isna(baseline_value):
    raise ValueError(
        "The baseline cell has no value, so a relative heatmap is undefined."
    )
print(f"Baseline ({BASELINE_STRUCTURE}) = {baseline_value:.2f}%")

delta_table = absolute_table - baseline_value
delta_limit = float(np.nanmax(np.abs(delta_table.to_numpy())))
show_heatmap(
    delta_table,
    f"Difference from the {BASELINE_STRUCTURE} baseline",
    "Survival difference (pp)",
    "Readout (v)",
    "Substation edges / nodes (e, n)",
    colorscale="RdBu",
    zmin=-delta_limit,
    zmax=delta_limit,
    zmid=0.0,
    text_format="+.1f",
)
print("done")

Baseline (e0n0v0) = 94.26%
Difference from the e0n0v0 baseline


readout,v0,v1
substation edges / nodes,,
e0n0,0.00,0.30
e0n1,0.00,-29.06
e1n0,2.35,-4.58
e1n1,-9.24,-27.73


done


In [61]:
show_heatmap(
    structure_table(heatmap_endpoint_df, "run_name", aggfunc="nunique"),
    "Runs contributing to each cell",
    "Runs",
    "Readout (v)",
    "Substation edges / nodes (e, n)",
    colorscale="Blues",
    zmin=0,
    text_format="d",
)

show_heatmap(
    structure_table(heatmap_endpoint_df, HEATMAP_VALUE_COLUMN, aggfunc="std"),
    "Seed-to-seed standard deviation per cell",
    "Std across seeds (pp)",
    "Readout (v)",
    "Substation edges / nodes (e, n)",
    colorscale="Oranges",
    zmin=0,
)
print("done")

Runs contributing to each cell


readout,v0,v1
substation edges / nodes,,
e0n0,3,3
e0n1,3,3
e1n0,3,3
e1n1,3,3


Seed-to-seed standard deviation per cell


readout,v0,v1
substation edges / nodes,,
e0n0,4.45,3.74
e0n1,0.98,0.35
e1n0,3.38,8.25
e1n1,6.55,30.47


done


### Heatmap with the individual seed values

Same cells and same colour scale as the full-factorial heatmap, but each cell
also prints its three per-seed values underneath the mean. This is the quickest
way to tell a real difference from one lucky seed: a cell whose mean is high
only because one seed is far above the other two is not a finding.

Seeds are labelled explicitly (`s0`, `s1`, `s2`) rather than positionally, so a
missing seed is visible instead of silently shifting the order.

In [62]:
def heatmap_text_color(value, vmin, vmax):
    """White on the dark end of the scale, near-black on the light end."""
    if pd.isna(value):
        return "#111827"
    span = (vmax - vmin) or 1.0
    return "#f9fafb" if (value - vmin) / span < 0.55 else "#111827"


def show_heatmap_with_seeds(
    frame,
    value_column,
    index_column,
    column_column,
    index_order,
    column_order,
    title,
    x_label,
    y_label,
    color_label,
    colorscale="Viridis",
    zmin=0,
    zmax=100,
    seed_column="seed",
    mean_font_size=22,
    seed_font_size=11,
    width=980,
):
    mean_table = frame.pivot_table(
        index=index_column,
        columns=column_column,
        values=value_column,
        aggfunc="mean",
    ).reindex(index=index_order, columns=column_order)
    if mean_table.notna().sum().sum() == 0:
        print(f"No data available for: {title}")
        return None

    seed_labels = {}
    for (row_key, column_key), group in frame.groupby(
        [index_column, column_column]
    ):
        ordered = group.sort_values(seed_column)
        seed_labels[(row_key, column_key)] = "   ".join(
            f"s{int(seed)} {value:.1f}"
            for seed, value in zip(
                ordered[seed_column], ordered[value_column]
            )
        )

    figure = go.Figure(
        go.Heatmap(
            z=mean_table.to_numpy(),
            x=[str(value) for value in mean_table.columns],
            y=[str(value) for value in mean_table.index],
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            colorbar={"title": color_label},
            hovertemplate=(
                f"{y_label}: %{{y}}<br>{x_label}: %{{x}}<br>"
                f"mean {color_label}: %{{z:.2f}}<extra></extra>"
            ),
        )
    )
    for row_index, row_key in enumerate(mean_table.index):
        for column_index, column_key in enumerate(mean_table.columns):
            value = mean_table.iloc[row_index, column_index]
            if pd.isna(value):
                continue
            color = heatmap_text_color(value, zmin, zmax)
            figure.add_annotation(
                x=str(column_key),
                y=str(row_key),
                text=f"<b>{value:.1f}</b>",
                showarrow=False,
                yshift=15,
                font={"size": mean_font_size, "color": color},
            )
            label = seed_labels.get((row_key, column_key))
            if label:
                figure.add_annotation(
                    x=str(column_key),
                    y=str(row_key),
                    text=label,
                    showarrow=False,
                    yshift=-16,
                    font={"size": seed_font_size, "color": color},
                )

    figure.update_layout(
        title=title,
        xaxis_title=x_label,
        yaxis_title=y_label,
        width=width,
        height=150 * len(mean_table.index) + 200,
        # Match px.imshow, which puts the first row at the top.
        yaxis={"autorange": "reversed"},
    )
    figure.show()
    return mean_table


seeded_absolute_table = show_heatmap_with_seeds(
    heatmap_endpoint_df,
    HEATMAP_VALUE_COLUMN,
    "en_label",
    "v_label",
    EN_ORDER,
    V_ORDER,
    "Full structural factorial — cell mean with the three seed values",
    "Readout (v)",
    "Substation edges / nodes (e, n)",
    "Mean survival (%)",
    colorscale="Viridis",
    zmin=0,
    zmax=100,
)
display(seeded_absolute_table.round(2))
print("done")

v_label,v0,v1
en_label,,
e0n0,94.26,94.56
e0n1,94.26,65.20
e1n0,96.61,89.68
e1n1,85.02,66.53


done


### Pairwise marginal heatmaps

Each of these averages over the third factor, so they are easier to read than
the full table but hide any three-way interaction. Use them together with the
eight-cell table above.

In [64]:
FACTOR_PAIRS = [
    ("substation_edges", "substation_nodes", "e", "n"),
    ("substation_edges", "virtual_node", "e", "v"),
    ("substation_nodes", "virtual_node", "n", "v"),
]

for row_column, column_column, row_code, column_code in FACTOR_PAIRS:
    table = heatmap_endpoint_df.pivot_table(
        index=row_column,
        columns=column_column,
        values=HEATMAP_VALUE_COLUMN,
        aggfunc="mean",
    ).reindex(index=[False, True], columns=[False, True])
    table.index = [f"{row_code}0", f"{row_code}1"]
    table.columns = [f"{column_code}0", f"{column_code}1"]
    table = table.rename_axis(index=row_column, columns=column_column)
    show_heatmap(
        table,
        f"Marginal {row_code} × {column_code} "
        f"(averaged over the third factor)",
        "Mean survival (%)",
        column_column,
        row_column,
        colorscale="Viridis",
        zmin=0,
        zmax=100,
    )
print("done")

Marginal e × n (averaged over the third factor)


substation_nodes,n0,n1
substation_edges,,
e0,94.41,79.73
e1,93.15,75.78


Marginal e × v (averaged over the third factor)


virtual_node,v0,v1
substation_edges,,
e0,94.26,79.88
e1,90.82,78.11


Marginal n × v (averaged over the third factor)


virtual_node,v0,v1
substation_nodes,,
n0,95.44,92.12
n1,89.64,65.87


done


### The factorial as a cube

`e`, `n`, and `v` vary independently, so the design is literally a cube: the
eight vertices are the eight structures and each of the twelve edges is a
single-factor flip with the other two held fixed. This shows all eight cells at
once with **no marginalising**, which the pairwise 2-D heatmaps above cannot do.

A `go.Volume`/`go.Isosurface` view is deliberately not used: with only two
levels per axis there is nothing meaningful to interpolate between, and a smooth
density would invent structure that the experiment never measured.

In [65]:
import itertools

cube_means = heatmap_endpoint_df.groupby(FACTOR_COLUMNS)[
    HEATMAP_VALUE_COLUMN
].mean()
cube_counts = heatmap_endpoint_df.groupby(FACTOR_COLUMNS)["run_name"].nunique()

vertex_rows = []
for edges, nodes, virtual in itertools.product([False, True], repeat=3):
    key = (edges, nodes, virtual)
    structure = f"e{int(edges)}n{int(nodes)}v{int(virtual)}"
    vertex_rows.append(
        {
            "structure": structure,
            "e": int(edges),
            "n": int(nodes),
            "v": int(virtual),
            "value": float(cube_means.get(key, np.nan)),
            "runs": int(cube_counts.get(key, 0)),
        }
    )
vertex_df = pd.DataFrame(vertex_rows)
cube_lookup = dict(zip(vertex_df["structure"], vertex_df["value"]))

missing_vertices = vertex_df.loc[vertex_df["value"].isna(), "structure"].tolist()
if missing_vertices:
    print("Vertices with no qualifying runs:", missing_vertices)


def show_structure_cube(
    frame,
    value_column,
    title,
    color_label,
    colorscale="Viridis",
    cmin=None,
    cmax=None,
    cmid=None,
    value_format="{:.1f}",
):
    plotted = frame.dropna(subset=[value_column])
    if plotted.empty:
        print(f"No data available for: {title}")
        return None

    # Twelve edges: vertex pairs differing in exactly one factor.
    edge_x, edge_y, edge_z = [], [], []
    for left, right in itertools.combinations(frame.itertuples(), 2):
        distance = (
            abs(left.e - right.e) + abs(left.n - right.n) + abs(left.v - right.v)
        )
        if distance == 1:
            edge_x += [left.e, right.e, None]
            edge_y += [left.n, right.n, None]
            edge_z += [left.v, right.v, None]

    figure = go.Figure()
    figure.add_trace(
        go.Scatter3d(
            x=edge_x,
            y=edge_y,
            z=edge_z,
            mode="lines",
            line={"color": "#9ca3af", "width": 3},
            hoverinfo="skip",
            showlegend=False,
        )
    )
    figure.add_trace(
        go.Scatter3d(
            x=plotted["e"],
            y=plotted["n"],
            z=plotted["v"],
            mode="markers+text",
            marker={
                "size": 22,
                "color": plotted[value_column],
                "colorscale": colorscale,
                "cmin": cmin,
                "cmax": cmax,
                "cmid": cmid,
                "opacity": 0.95,
                "line": {"color": "#111827", "width": 1},
                "colorbar": {"title": color_label},
            },
            text=[
                f"{structure}<br>{value_format.format(value)}"
                for structure, value in zip(
                    plotted["structure"], plotted[value_column]
                )
            ],
            textposition="top center",
            customdata=np.stack(
                [plotted["structure"], plotted["runs"]], axis=-1
            ),
            hovertemplate=(
                "%{customdata[0]}<br>"
                + color_label
                + ": %{marker.color:.2f}<br>runs: %{customdata[1]}"
                "<extra></extra>"
            ),
            showlegend=False,
        )
    )
    # Ring the baseline vertex so it is findable while rotating.
    baseline_vertex = plotted[plotted["structure"] == BASELINE_STRUCTURE]
    if not baseline_vertex.empty:
        figure.add_trace(
            go.Scatter3d(
                x=baseline_vertex["e"],
                y=baseline_vertex["n"],
                z=baseline_vertex["v"],
                mode="markers",
                marker={
                    "size": 32,
                    "color": "rgba(0,0,0,0)",
                    "line": {"color": "#dc2626", "width": 5},
                },
                name=f"baseline {BASELINE_STRUCTURE}",
                hoverinfo="skip",
            )
        )

    axis = lambda label: {  # noqa: E731
        "title": label,
        "tickvals": [0, 1],
        "ticktext": ["off", "on"],
        "range": [-0.35, 1.35],
    }
    figure.update_layout(
        title=title,
        scene={
            "xaxis": axis("e: substation edges"),
            "yaxis": axis("n: substation nodes"),
            "zaxis": axis("v: virtual-node readout"),
            "camera": {"eye": {"x": 1.7, "y": 1.7, "z": 1.1}},
        },
        width=1000,
        height=760,
        margin={"l": 0, "r": 0, "t": 60, "b": 0},
        showlegend=True,
        legend={"orientation": "h", "y": -0.02},
    )
    figure.show()
    return figure


structure_cube_figure = show_structure_cube(
    vertex_df,
    "value",
    "gs_s2 structural cube — mean final-"
    f"{HEATMAP_LAST_N_TEST_EVALS} test survival (%)",
    "Mean survival (%)",
    colorscale="Viridis",
    cmin=0,
    cmax=100,
)
print("done")

done


In [66]:
vertex_delta_df = vertex_df.copy()
vertex_delta_df["delta"] = vertex_delta_df["value"] - baseline_value
cube_delta_limit = float(np.nanmax(np.abs(vertex_delta_df["delta"].to_numpy())))

structure_cube_delta_figure = show_structure_cube(
    vertex_delta_df,
    "delta",
    f"gs_s2 structural cube — difference from {BASELINE_STRUCTURE} (pp)",
    "Difference (pp)",
    colorscale="RdBu",
    cmin=-cube_delta_limit,
    cmax=cube_delta_limit,
    cmid=0.0,
    value_format="{:+.1f}",
)
print("done")

done


### Edge effects: where the interactions are

Each row is one **edge of the cube** -- flipping a single factor while the other
two stay fixed. For a factor with a clean main effect, its four deltas agree in
sign and rough size; if they disagree, the factor's usefulness depends on the
other two and the single main-effect number above is misleading.

`spread_pp` is the range of the four deltas for that factor, so the factor with
the largest spread is the one most entangled with the others.

> **Statistic note.** These edges use the heatmap statistic
> (`last_n_test_survival_pct`, the final-N unsmoothed mean) so they are directly
> comparable with the cube and the heatmaps. The main-effect table further down
> uses `comparison_survival_pct` at the common budget instead, so a factor's
> `mean_delta_pp` here will not exactly equal its `effect_pp` there. They answer
> the same question at two different measurement points; if they disagree in
> sign, trust neither until the coverage spread is understood.

In [67]:
edge_rows = []
for code_key, (column, _off_label, _on_label) in STRUCTURE_FACTORS.items():
    other_columns = [name for name in FACTOR_COLUMNS if name != column]
    other_codes = [
        key
        for key, (name, _o, _n) in STRUCTURE_FACTORS.items()
        if name != column
    ]
    for other_values in itertools.product([False, True], repeat=2):
        assignment = dict(zip(other_columns, other_values))

        def structure_for(flag, assignment=assignment, column=column):
            full = {**assignment, column: flag}
            return "e{}n{}v{}".format(
                int(full["substation_edges"]),
                int(full["substation_nodes"]),
                int(full["virtual_node"]),
            )

        off_structure = structure_for(False)
        on_structure = structure_for(True)
        off_value = cube_lookup.get(off_structure, np.nan)
        on_value = cube_lookup.get(on_structure, np.nan)
        edge_rows.append(
            {
                "factor": code_key,
                "held_fixed": " ".join(
                    f"{other_code}{int(value)}"
                    for other_code, value in zip(other_codes, other_values)
                ),
                "off_structure": off_structure,
                "on_structure": on_structure,
                "off_pct": off_value,
                "on_pct": on_value,
                "delta_pp": on_value - off_value,
            }
        )

edge_effects = pd.DataFrame(edge_rows)
print("Single-factor flips along each cube edge:")
display(edge_effects.round(2))

edge_summary = (
    edge_effects.groupby("factor", as_index=False)
    .agg(
        mean_delta_pp=("delta_pp", "mean"),
        min_delta_pp=("delta_pp", "min"),
        max_delta_pp=("delta_pp", "max"),
        edges=("delta_pp", "count"),
        sign_flips=("delta_pp", lambda x: int((x > 0).any() and (x < 0).any())),
    )
    .sort_values("mean_delta_pp", ascending=False)
)
edge_summary["spread_pp"] = (
    edge_summary["max_delta_pp"] - edge_summary["min_delta_pp"]
)
print(
    "Per-factor edge summary (sign_flips=1 means the factor helps in some "
    "contexts and hurts in others):"
)
display(edge_summary.round(2))

edge_plot = px.bar(
    edge_effects,
    x="delta_pp",
    y="held_fixed",
    color="factor",
    barmode="group",
    orientation="h",
    hover_data=["off_structure", "on_structure", "off_pct", "on_pct"],
    title="Effect of enabling each factor, conditioned on the other two",
    labels={
        "delta_pp": "Survival change when enabling the factor (pp)",
        "held_fixed": "Other two factors",
    },
    height=560,
)
edge_plot.add_vline(x=0.0, line_dash="dot")
edge_plot.show()
print("done")

Single-factor flips along each cube edge:


,factor,held_fixed,off_structure,on_structure,off_pct,on_pct,delta_pp
0,e,n0 v0,e0n0v0,e1n0v0,94.26,96.61,2.35
1,e,n0 v1,e0n0v1,e1n0v1,94.56,89.68,-4.88
2,e,n1 v0,e0n1v0,e1n1v0,94.26,85.02,-9.24
3,e,n1 v1,e0n1v1,e1n1v1,65.20,66.53,1.33
4,n,e0 v0,e0n0v0,e0n1v0,94.26,94.26,0.00
5,n,e0 v1,e0n0v1,e0n1v1,94.56,65.20,-29.36
6,n,e1 v0,e1n0v0,e1n1v0,96.61,85.02,-11.59
7,n,e1 v1,e1n0v1,e1n1v1,89.68,66.53,-23.15
8,v,e0 n0,e0n0v0,e0n0v1,94.26,94.56,0.30
9,v,e0 n1,e0n1v0,e0n1v1,94.26,65.20,-29.06


Per-factor edge summary (sign_flips=1 means the factor helps in some contexts and hurts in others):


,factor,mean_delta_pp,min_delta_pp,max_delta_pp,edges,sign_flips,spread_pp
0,e,-2.61,-9.24,2.35,4,1,11.59
2,v,-13.55,-29.06,0.30,4,1,29.36
1,n,-16.02,-29.36,0.00,4,1,29.36


done


## Structure ranking and main effects

The ranking is seed-aggregated at the common budget with the baseline delta
attached. The main-effect table then averages every run with a factor on
against every run with it off -- valid here because the design is a balanced
full factorial with equal seeds per cell.

In [68]:
valid_endpoint_df = endpoint_df.dropna(
    subset=["comparison_survival_pct"]
).copy()

ranking = (
    valid_endpoint_df.groupby("structure", as_index=False)
    .agg(
        mean_survival_pct=("comparison_survival_pct", "mean"),
        std_survival_pct=("comparison_survival_pct", "std"),
        min_survival_pct=("comparison_survival_pct", "min"),
        max_survival_pct=("comparison_survival_pct", "max"),
        seeds=("seed", "nunique"),
        backends=("compute_backend", lambda x: ", ".join(sorted(set(x)))),
    )
    .sort_values("mean_survival_pct", ascending=False)
)

baseline_mask = ranking["structure"].eq(BASELINE_STRUCTURE)
if not baseline_mask.any():
    raise ValueError("The baseline structure is missing from the ranking.")
baseline_mean = float(ranking.loc[baseline_mask, "mean_survival_pct"].iloc[0])
ranking["mean_minus_baseline"] = ranking["mean_survival_pct"] - baseline_mean

display(ranking.round(2))

rank_plot = px.bar(
    ranking.sort_values("mean_minus_baseline"),
    x="mean_minus_baseline",
    y="structure",
    orientation="h",
    color="mean_minus_baseline",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    hover_data=["mean_survival_pct", "std_survival_pct", "seeds"],
    title=(
        f"Survival at the {comparison_budget / 1_000_000:.3f}M-step budget, "
        f"relative to {BASELINE_STRUCTURE}"
    ),
    labels={
        "mean_minus_baseline": "Difference from baseline (pp)",
        "structure": "Structure",
    },
    height=480,
)
rank_plot.add_vline(x=0.0, line_dash="dot")
rank_plot.show()
print("done")

,structure,mean_survival_pct,std_survival_pct,min_survival_pct,max_survival_pct,seeds,backends,mean_minus_baseline
4,e1n0v0,96.22,3.69,91.98,98.70,3,"IZAR (GPU), JED (CPU)",1.79
2,e0n1v0,94.64,4.29,90.81,99.27,3,"IZAR (GPU), JED (CPU)",0.21
0,e0n0v0,94.43,4.45,89.83,98.70,3,"IZAR (GPU), JED (CPU)",0.00
1,e0n0v1,92.62,6.26,85.39,96.36,3,"IZAR (GPU), JED (CPU)",-1.81
5,e1n0v1,89.45,6.99,82.28,96.24,3,"IZAR (GPU), JED (CPU)",-4.98
6,e1n1v0,83.72,8.82,73.55,89.32,3,"IZAR (GPU), JED (CPU)",-10.71
7,e1n1v1,67.54,32.28,30.88,91.71,3,"IZAR (GPU), JED (CPU)",-26.88
3,e0n1v1,62.01,6.68,54.30,65.94,3,"IZAR (GPU), JED (CPU)",-32.42


done


In [69]:
main_effect_rows = []
for code_key, (column, off_label, on_label) in STRUCTURE_FACTORS.items():
    off = valid_endpoint_df[~valid_endpoint_df[column]]
    on = valid_endpoint_df[valid_endpoint_df[column]]
    if off.empty or on.empty:
        print(f"Skipping factor {code_key}: one level has no completed runs.")
        continue
    off_mean = float(off["comparison_survival_pct"].mean())
    on_mean = float(on["comparison_survival_pct"].mean())
    main_effect_rows.append(
        {
            "factor": code_key,
            "column": column,
            "off_mean_pct": off_mean,
            "on_mean_pct": on_mean,
            "effect_pp": on_mean - off_mean,
            "off_std": float(off["comparison_survival_pct"].std()),
            "on_std": float(on["comparison_survival_pct"].std()),
            "runs_off": int(off["run_name"].nunique()),
            "runs_on": int(on["run_name"].nunique()),
        }
    )

main_effects = pd.DataFrame(main_effect_rows).sort_values(
    "effect_pp", ascending=False
)
print("Main effects (positive means enabling the factor helped):")
display(main_effects.round(2))
print("done")

Main effects (positive means enabling the factor helped):


,factor,column,off_mean_pct,on_mean_pct,effect_pp,off_std,on_std,runs_off,runs_on
0,e,substation_edges,85.92,84.23,-1.69,15.19,18.37,12,12
2,v,virtual_node,92.25,77.91,-14.34,7.11,20.18,12,12
1,n,substation_nodes,93.18,76.98,-16.20,5.38,19.95,12,12


done


## Optional CSV export

In [70]:
EXPORT_TABLES = False

if EXPORT_TABLES:
    export_dir = wm.TASK_DIR / "outputs" / "gs_s2_structure_summary"
    export_dir.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(export_dir / "coverage.csv", index=False)
    endpoint_df.to_csv(export_dir / "run_endpoints.csv", index=False)
    ranking.to_csv(export_dir / "structure_ranking.csv", index=False)
    main_effects.to_csv(export_dir / "main_effects.csv", index=False)
    absolute_table.to_csv(export_dir / "heatmap_absolute.csv")
    delta_table.to_csv(export_dir / "heatmap_baseline_delta.csv")
    print("Saved tables under", export_dir)
else:
    print("Set EXPORT_TABLES = True to write CSVs.")
print("done")

Set EXPORT_TABLES = True to write CSVs.
done
